In [1]:
### Build a and a^\dagger

import numpy as np
### AMximum modes for each site is d. Here we will restrict to d=4
d = 4

# Creation operator a†
adag = np.zeros((d, d))
for n in range(d - 1):
    adag[n+1, n] = np.sqrt(n+1)

# Annihilation operator a
a = adag.T

# Parameters (you can tune later)
m = 1.0

# Field operators
phi = (a + adag) / np.sqrt(2 * m)
pi  = 1j * np.sqrt(m / 2) * (adag - a)

# Define 
phi2 = phi @ phi
phi4 = phi2 @ phi2
pi2  = pi @ pi

#Define interation part
phi_i = embed_operator(phi_op, i, num_sites)
phi_i2 = phi_i @ phi_i

phi_j = embed_operator(phi_op, i+1, num_sites)
phi_j2 = phi_j @ phi_j

interaction = phi_i @ phi_j

H += 1/(2*a**2)(phi_i2 + phi_j2 - 2 * interaction)

In [2]:
from qiskit.quantum_info import Operator, SparsePauliOp

def to_pauli_op(matrix):
    op = Operator(matrix)
    return SparsePauliOp.from_operator(op)

phi_op  = to_pauli_op(phi)
pi_op   = to_pauli_op(pi)
phi2_op = to_pauli_op(phi2)
phi4_op = to_pauli_op(phi4)
pi2_op  = to_pauli_op(pi2)

In [3]:
## Embed into N site lattice.. Total d*N qubit

def embed_operator(op, site, num_sites):
    """
    op: SparsePauliOp acting on 2 qubits
    site: which site (0 to num_sites-1)
    num_sites: total sites
    """
    total_qubits = 2 * num_sites
    start = 2 * site  # each site = 2 qubits
    
    full_op = None

    for pauli, coeff in zip(op.paulis, op.coeffs):
        label = list("I" * total_qubits)

        # place 2-qubit operator
        for i, p in enumerate(pauli.to_label()):
            label[start + i] = p

        new_pauli = "".join(label)
        term = SparsePauliOp.from_list([(new_pauli, coeff)])

        full_op = term if full_op is None else full_op + term

    # return full_op
    
    # for i in range(num_sites):
    #     if i == site:
    #         current = op
    #     else:
    #         current = SparsePauliOp.from_list([("II", 1.0)])
        
    #     full_op = current if full_op is None else full_op.tensor(current)
    
    return full_op

In [4]:
## Build free phi^4 Hamiltonian 
num_sites = 5
lam = 1.0
m2=-1
H = None

# --- Local terms ---
for n in range(num_sites):
    H_pi = embed_operator(pi2_op, n, num_sites) * 0.5
    H_m  = embed_operator(phi2_op, n, num_sites) * (0.5 * m2)
    H_l  = embed_operator(phi4_op, n, num_sites) * (lam / 24)

    term = H_pi + H_m + H_l
    H = term if H is None else H + term

In [5]:
#Calculate ground state exactly
H_local = 0.5 * pi2 + 0.5 * m2 * phi2 + (lam / 24) * phi4
evals, evecs = np.linalg.eigh(H)
print (evals)
E_exact = np.min(evals)

print("Exact ground state energy:", E_exact)

[-4.57700881 -4.13657611 -4.13657611 ...  5.69907611  5.69907611
  6.13950881]
Exact ground state energy: -4.5770088123382


In [18]:
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms import VQE
from qiskit_aer.primitives import EstimatorV2
from qiskit_aer import AerSimulator
from qiskit_algorithms.optimizers import SPSA
from qiskit import transpile




backend = AerSimulator()

shots = 4096
estimator = EstimatorV2(
    options={
        "run_options": {
            "shots": 16384,   # high for stability
            "seed": 42        # reproducible
        }
    }
)
ansatz = TwoLocal(
    num_qubits=10,
    rotation_blocks=['ry', 'rz'],
    entanglement_blocks='cx',
    reps=2,
    entanglement='full'
)
optimizer = SPSA(maxiter=200)

ansatz = transpile(ansatz, backend)



vqe = VQE(estimator, ansatz, optimizer)

result = vqe.compute_minimum_eigenvalue(H)

E_vqe = result.eigenvalue.real

print("VQE ground state energy:", E_vqe)





/var/folders/g6/7wnpvdrx3pq6vgv_ykqncp7w0000gn/T/ipykernel_72212/2220722228.py:22: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(


VQE ground state energy: 0.35982377795501724


In [5]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

In [6]:
#Parameters
num_sites = 5          # lattice size
d = 4                  # truncation (0,1,2,3)
num_qubits = 2 * num_sites

m2 = -1.0              # mass^2 (negative → SSB)
lam = 1.0              # lambda
a = 1.0                # lattice spacing
J = 1/(2*a**2)         # gradient coupling

In [7]:
# Creation operator a†
adag_op = np.zeros((d, d))
for n in range(d - 1):
    adag_op[n+1, n] = np.sqrt(n+1)
    
# Annihilation operator a
a_op = adag_op.T

# frequency (can tune)
omega = 1.0

# field operator
phi_mat = (a_op + adag_op) / np.sqrt(2*omega)

# momentum operator
pi_mat = 1j * (adag_op - a_op) * np.sqrt(omega/2)

In [8]:
phi2_mat = phi_mat @ phi_mat
phi4_mat = phi2_mat @ phi2_mat
pi2_mat  = pi_mat @ pi_mat

In [9]:
from qiskit.quantum_info import Operator

def matrix_to_pauli(mat):
    op = Operator(mat)
    return SparsePauliOp.from_operator(op)

In [10]:
phi_op  = matrix_to_pauli(phi_mat)
phi2_op = matrix_to_pauli(phi2_mat)
phi4_op = matrix_to_pauli(phi4_mat)
pi2_op  = matrix_to_pauli(pi2_mat)

In [11]:
def embed_operator(op, site, num_sites):
    total_qubits = 2 * num_sites
    start = 2 * site

    full_op = None

    for pauli, coeff in zip(op.paulis, op.coeffs):
        label = list("I" * total_qubits)

        for i, p in enumerate(pauli.to_label()):
            label[start + i] = p

        new_pauli = "".join(label)
        term = SparsePauliOp.from_list([(new_pauli, coeff)])

        full_op = term if full_op is None else full_op + term

    return full_op

In [12]:
H = 0

for i in range(num_sites):
    H_pi = 0.5 * embed_operator(pi2_op, i, num_sites)
    H_m  = 0.5 * m2 * embed_operator(phi2_op, i, num_sites)
    H_l  = (lam / 24) * embed_operator(phi4_op, i, num_sites)

    H += H_pi + H_m + H_l

In [13]:
for i in range(num_sites - 1):
    phi_i = embed_operator(phi_op, i, num_sites)
    phi_j = embed_operator(phi_op, i+1, num_sites)

    H += J * (phi_i - phi_j) @ (phi_i - phi_j)

In [14]:
H = H.simplify()

In [15]:
#Calculate ground state exactly
H_local = 0.5 * pi2_mat + 0.5 * m2 * phi2_mat + (lam / 24) * phi4_mat
evals, evecs = np.linalg.eigh(H)
print (evals)
E_exact = np.min(evals)

print("Exact ground state energy:", E_exact)

[-2.91314104 -2.91309781 -0.99924539 ... 17.82738625 19.59130221
 19.59221871]
Exact ground state energy: -2.9131410368324877


In [16]:
from qiskit.circuit.library import TwoLocal

ansatz = TwoLocal(
    num_qubits=num_qubits,
    rotation_blocks="ry",
    entanglement_blocks="cz",
    reps=2
)

/var/folders/g6/7wnpvdrx3pq6vgv_ykqncp7w0000gn/T/ipykernel_72212/3579645273.py:3: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(
